# Basis generality: order sweep under a B-spline tensor basis
Generates results/bspline_basis/. Same frozen select_order, design constructor swapped from polynomials to a cubic B-spline tensor basis; both bases run on the same seeded data per replicate.

In [ ]:
import numpy as np, json, csv, time, hashlib, zlib, inspect, sys, platform
import scipy
from itertools import product as iproduct, combinations
from scipy import stats
from scipy.interpolate import BSpline
import os
OUT = '/content/drive/MyDrive/ORDER_SWEEP/results/bspline_basis'; os.makedirs(OUT, exist_ok=True)

def poly_design(X, k, D=4):
    exps=[c for c in iproduct(range(D+1),repeat=X.shape[1]) if sum(c)<=D and sum(1 for x in c if x>0)<=k]
    cols=[]
    for e in exps:
        col=np.ones(X.shape[0])
        for j,p in enumerate(e):
            if p>0: col=col*X[:,j]**p
        cols.append(col)
    return np.column_stack(cols), exps.index(tuple([0]*X.shape[1]))
def bspline_1d(xj, degree=3, n_interior=1):
    lo,hi=float(xj.min()),float(xj.max()); span=hi-lo+1.0
    qs=np.quantile(xj,np.linspace(0,1,n_interior+2)[1:-1]) if n_interior>0 else np.array([])
    eps=1e-6*span; t=np.r_[[lo-eps]*(degree+1),qs,[hi+eps]*(degree+1)]
    xc=np.clip(xj,lo-eps/2,hi+eps/2)
    B=np.asarray(BSpline.design_matrix(xc,t,degree).todense())
    return B[:,1:]
def bspline_design(X, k, degree=3, n_interior=1):
    n,d=X.shape; Bs=[bspline_1d(X[:,j],degree,n_interior) for j in range(d)]
    cols=[np.ones(n)]
    for m in range(1,k+1):
        for S in combinations(range(d),m):
            blocks=[Bs[j] for j in S]
            for idxs in iproduct(*[range(b.shape[1]) for b in blocks]):
                col=np.ones(n)
                for b,ii in zip(blocks,idxs): col=col*b[:,ii]
                cols.append(col)
    return np.column_stack(cols), 0
def select_order(X,h,design_fn,K=3,S=10,alpha=0.05,train_frac=0.75):
    n=X.shape[0]; n_tr=int(train_frac*n); n_te=n-n_tr
    designs={k:design_fn(X,k) for k in range(1,K+1)}; r2={}
    for s in range(S):
        idx=np.random.default_rng(s).permutation(n); tr,te=idx[:n_tr],idx[n_tr:]; out={}
        for k in range(1,K+1):
            P0,ci=designs[k]; mu=P0[tr].mean(0); sd=P0[tr].std(0); sd[sd==0]=1.0
            P=(P0-mu)/sd; P[:,ci]=1.0; hm=h[tr].mean()
            beta,*_=np.linalg.lstsq(P[tr],h[tr]-hm,rcond=None); resid=(h[te]-hm)-P[te]@beta
            out[k]=1.0-float((resid@resid)/np.sum((h[te]-h[te].mean())**2))
        r2[s]=out
    corr=1.0/S+n_te/n_tr; pf={k:designs[k][0].shape[1] for k in range(1,K+1)}
    om=float(np.mean([1.0-r2[s][K] for s in range(S)])); stat={}
    for k in range(1,K):
        g=np.array([r2[s][K]-r2[s][k] for s in range(S)]); m=g.mean(); v=g.var(ddof=1)*corr
        opt=(pf[K]-pf[k])*om/n_tr
        if v>0: t=m/np.sqrt(v); p=1.0-stats.t.cdf(t,df=S-1); ub=m+stats.t.ppf(1-alpha,df=S-1)*np.sqrt(v)+opt
        else: p=0.0 if m>0 else 1.0; ub=m+opt
        stat[k]={"mean":float(m),"p":float(p),"ub":float(ub)}
    khat=K
    for k in range(1,K):
        if stat[k]["p"]>alpha: khat=k; break
    return khat,stat,None
def make_X(n,dep,rng):
    Z=rng.standard_normal((n,3))
    if dep.startswith("pair"):
        rho=float(dep[4:]); Z[:,2]=rho*Z[:,0]+np.sqrt(1-rho**2)*Z[:,2]
    return Z
def make_h(X,dgp,sig,rng):
    x1,x2,x3=X.T
    f={"o1":x1+np.tanh(x2)-0.5*x3,"o2":x1*x2+np.tanh(x3),"o3":x1*x2*x3,"o3mix":x1*x2*x3+0.5*x1*x2}[dgp]
    return f+sig*rng.standard_normal(len(f))
TRUE={"o1":1,"o2":2,"o3":3,"o3mix":3}
N,R=20_000,20; DGPS=["o1","o2","o3","o3mix"]; DEPS=["indep","pair0.9"]; SIGS=[0.0,0.5]
rows=[]
for dgp in DGPS:
  for dep in DEPS:
    for sig in SIGS:
      for rep in range(R):
        seed=(70_000+zlib.crc32(f"bspline|{dgp}|{dep}|{sig}".encode())+rep*977)%2**32
        rng=np.random.default_rng(seed); X=make_X(N,dep,rng); h=make_h(X,dgp,sig,rng)
        kb,sb,_=select_order(X,h,bspline_design,K=3)
        kp,sp,_=select_order(X,h,poly_design,K=3)
        rows.append({"experiment":"bspline_basis","dgp":dgp,"dependence":dep,"sigma":sig,"rep":rep,
                     "true_order":TRUE[dgp],"khat_bspline":kb,"khat_poly":kp,
                     "rem1_p_bspline":sb[1]["p"],"rem2_p_bspline":sb[2]["p"]})
with open(os.path.join(OUT,"per_seed_bspline.csv"),"w",newline="") as f:
    w=csv.DictWriter(f,fieldnames=list(rows[0].keys())); w.writeheader(); w.writerows(rows)
print("wrote", len(rows), "rows to bspline_basis/per_seed_bspline.csv")
